In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# -------------------------------
# Sample Data
# -------------------------------
pairs = [
    ("hello", "hi how are you"),
    ("how are you", "i am fine"),
    ("what is your name", "i am a chatbot"),
    ("bye", "goodbye")
]

# -------------------------------
# Preprocessing
# -------------------------------
def tokenize(sentence):
    return sentence.lower().split()

vocab = set()
for inp, out in pairs:
    vocab.update(tokenize(inp))
    vocab.update(tokenize(out))

word2idx = {word: i+2 for i, word in enumerate(vocab)}
word2idx["<pad>"] = 0
word2idx["<sos>"] = 1
idx2word = {i: w for w, i in word2idx.items()}

vocab_size = len(word2idx)

def encode(sentence):
    return [word2idx[w] for w in tokenize(sentence)]

data = [(encode(inp), encode(out)) for inp, out in pairs]

# -------------------------------
# Encoder
# -------------------------------
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell

# -------------------------------
# Attention
# -------------------------------
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.shape[1]

        hidden = hidden[-1].unsqueeze(1).repeat(1, seq_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        return torch.softmax(attention, dim=1)

# -------------------------------
# Decoder (FIXED)
# -------------------------------
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(hidden_dim + emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden, cell, context):
        # ✅ Ensure x is always [batch_size]
        if x.dim() == 0:
            x = x.unsqueeze(0)

        x = x.unsqueeze(1)  # [batch_size, 1]
        embedded = self.embedding(x)

        context = context.unsqueeze(1)
        lstm_input = torch.cat((embedded, context), dim=2)

        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        prediction = self.fc(output.squeeze(1))

        return prediction, hidden, cell

# -------------------------------
# Seq2Seq Model
# -------------------------------
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, attention):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.attention = attention

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        trg_len = len(trg)
        outputs = torch.zeros(trg_len, vocab_size)

        encoder_outputs, hidden, cell = self.encoder(src)

        input_token = torch.tensor([word2idx["<sos>"]])

        for t in range(trg_len):
            attn_weights = self.attention(hidden, encoder_outputs)
            context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)

            output, hidden, cell = self.decoder(input_token, hidden, cell, context)
            outputs[t] = output

            top1 = output.argmax(1)

            # ✅ FIXED teacher forcing
            if np.random.rand() < teacher_forcing_ratio:
                input_token = trg[t].unsqueeze(0)
            else:
                input_token = top1

        return outputs

# -------------------------------
# Initialize Model
# -------------------------------
EMB_DIM = 64
HIDDEN_DIM = 128

encoder = Encoder(vocab_size, EMB_DIM, HIDDEN_DIM)
attention = Attention(HIDDEN_DIM)
decoder = Decoder(vocab_size, EMB_DIM, HIDDEN_DIM)

model = Seq2Seq(encoder, decoder, attention)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# -------------------------------
# Training
# -------------------------------
epochs = 300

for epoch in range(epochs):
    total_loss = 0

    for src, trg in data:
        src_tensor = torch.tensor([src])
        trg_tensor = torch.tensor(trg)

        optimizer.zero_grad()
        output = model(src_tensor, trg_tensor)

        loss = 0
        for t in range(len(trg)):
            loss += criterion(output[t].unsqueeze(0), trg_tensor[t].unsqueeze(0))

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

# -------------------------------
# Chat Function
# -------------------------------
def chat(sentence):
    model.eval()
    with torch.no_grad():
        src = torch.tensor([encode(sentence)])
        encoder_outputs, hidden, cell = encoder(src)

        input_token = torch.tensor([word2idx["<sos>"]])
        result = []

        for _ in range(10):
            attn_weights = attention(hidden, encoder_outputs)
            context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)

            output, hidden, cell = decoder(input_token, hidden, cell, context)
            top1 = output.argmax(1).item()

            if top1 == 0:
                break

            result.append(idx2word.get(top1, "<unk>"))
            input_token = torch.tensor([top1])

        return " ".join(result)

# -------------------------------
# Test
# -------------------------------
print("\nChatbot Testing:")
print("Input: how are you")
print("Output:", chat("how are you"))

Epoch 0, Loss: 34.6225
Epoch 50, Loss: 0.1445
Epoch 100, Loss: 0.0471
Epoch 150, Loss: 0.0243
Epoch 200, Loss: 0.0151
Epoch 250, Loss: 0.0104

Chatbot Testing:
Input: how are you
Output: i am fine fine fine fine fine fine fine fine
